# Required Packages


# Corpus Building

## Raw corpora scrapping

In [2]:
# 增强版代码：一步到位
import requests #用于向网站发送请求
import json 
import time  #用于对请求加延时，爬取速度太快容易被反爬
from pathlib import Path 
import re


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36", # 模擬瀏覽器
    "Referer": "http://www.npc.gov.cn/npc/c191/c12481/search/index.html", # 搜尋界面的網址URL 
    "Accept": "application/json, text/plain, */*", 
}
SEARCH_API = "http://www.npc.gov.cn/search" # 從瀏覽器開發者介面中取得 

PROJ_PATH = Path("01_MATh")

OUTPUT = "raw_list.jsonl"
OUTPUT_New = "fined_list.jsonl"
SLEEP = 0.5

QUERIES = [
    "全过程人民民主",
    "全过程民主",
    "基层立法联系点",
    "立法联系点",
]
COMPILED = [re.compile(p) for p in QUERIES] #用于精确匹配标题然后筛选

START = "2019-11-02 00:00:00" 
END   = "2026-05-31 23:59:59" 

# 测试一下只用 最少的全过程民主关键词
# QUERIES = [ "全过程民主" ] 
def search_page(keyword, page_num, page_size=50,
                start_time="", end_time=""):
    params = {
        "allKeywords": keyword,
        "siteId": 5,
        "searchTag": 1,
        "sort": "",
        "position": 1,
        "startTime": start_time,
        "endTime": end_time,
        "pageSize": page_size,
        "pageNum": page_num,
    }
    r = requests.get(SEARCH_API, params=params, headers=HEADERS, timeout=20)
    r.raise_for_status()

    #### 处理异常信息（阶段1：识别问题）#####
    resp = r.json()
    if not isinstance(resp.get("data"), dict): #增加行：打印异常的完整响应
        print(f"  [ANOMALY] page={page_num} resp={resp}")  #增加行：打印异常的完整响应
    #### 处理异常信息（阶段1: 识别问题） ####



    return resp 

def collect_one_query(keyword, page_size=50, start_time="", end_time=""):
    items = []
    page = 1
    while True:
        try:
            resp = search_page(keyword, page, page_size, start_time, end_time)
        except Exception as e:
            print(f"  [ERR] {e},重试")
            time.sleep(5)
            try:
                resp = search_page(keyword, page, page_size, start_time, end_time)
            except Exception as e2:
                print(f"  [ERR] 二次失败,跳过 page {page}: {e2}")
                page += 1
                continue
        if resp.get("state") != 200:
            print(f"  [WARN] state={resp.get('state')} msg={resp.get('msg')}")
            break
        block = resp.get("data", {})
                
        # 09.16 实装 测试能不能用它绕过 str has no attribute 'get'  ↓↓↓
        try:
             total = block.get("total", 0) # BUG AttributeError: 'str' object has no attribute 'get'
        except Exception as e3:
            print(f" [ERR] 经典报错，跳过page {page}:{e3}")
            page += 1
            # time.sleep(SLEEP) 
            time.sleep(5) #实测会一旦经典报错就会连续报错，试试多睡一会儿。
            continue 
        # 09.16 实装 测试能不能用它绕过 str has no attribute 'get'  ↑↑↑

        
        # total = block.get("total", 0) # BUG AttributeError: 'str' object has no attribute 'get'
        batch = block.get("data", []) or []
        if page == 1:
            print(f"  服务端 total = {total}")
        if not batch:
            break
        items.extend(batch)
        print(f"  page {page}: +{len(batch)},累计 {len(items)}/{total}")
        if len(items) >= total or len(batch) < page_size:
            break
        page += 1
        time.sleep(SLEEP)
        if page > 1000:
            print("  [SAFETY] 超 1000 页,中断")
            break
    return items


def main():
    with open(OUTPUT_New, "w", encoding="utf-8") as fout:
        seen = set()

        for q in QUERIES:
            print(f"\n=== 检索词: {q} ===")
            raw = collect_one_query(q) # raw是一个列表，列表中的元素是字典
            for it in raw: # it 是 每一行 jsonl ，结构是一个字典
                date = str(it.get("docpubtime", ""))[:10]
                if not (START <= date <= END):
                    continue
                title = it.get("title","") or ""
                if not any(r.search(title) for r in COMPILED):
                    continue
                content = it.get("news_doccontent")
                # 修改判断逻辑，从判断标题改为判断正文 ↓↓↓
                if content and content.strip() and content not in seen:  # 同时排除内容为空或者内容重复的情况
                # 修改判断逻辑，从判断标题改为判断正文 ↑↑↑
                    seen.add(content)
                    rec = {
                        "date":  date,
                        "title": title,
                        "url":   it.get("docpuburl"),
                        "content": it.get("news_doccontent"),
                    }
                    rec["__query"] = q   # 在每条上贴上"是哪个检索词捞到的"
                    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

            fout.flush()
    print(f"\n完成,精炼数据落，2026年9月16日版。 {OUTPUT_New}")



main()


=== 检索词: 全过程人民民主 ===
  服务端 total = 10000
  page 1: +50,累计 50/10000
  page 2: +50,累计 100/10000
  page 3: +50,累计 150/10000
  page 4: +50,累计 200/10000
  page 5: +50,累计 250/10000
  page 6: +50,累计 300/10000
  page 7: +50,累计 350/10000
  page 8: +50,累计 400/10000
  page 9: +50,累计 450/10000
  page 10: +50,累计 500/10000
  page 11: +50,累计 550/10000
  page 12: +50,累计 600/10000
  page 13: +50,累计 650/10000
  page 14: +50,累计 700/10000
  page 15: +50,累计 750/10000
  page 16: +50,累计 800/10000
  page 17: +50,累计 850/10000
  page 18: +50,累计 900/10000
  page 19: +50,累计 950/10000
  page 20: +50,累计 1000/10000
  page 21: +50,累计 1050/10000
  page 22: +50,累计 1100/10000
  page 23: +50,累计 1150/10000
  page 24: +50,累计 1200/10000
  page 25: +50,累计 1250/10000
  page 26: +50,累计 1300/10000
  page 27: +50,累计 1350/10000
  page 28: +50,累计 1400/10000
  page 29: +50,累计 1450/10000
  page 30: +50,累计 1500/10000
  page 31: +50,累计 1550/10000
  page 32: +50,累计 1600/10000
  page 33: +50,累计 1650/10000
  page 34: +50,累计 1700/10000
 

## store each JSON Line as a single text file


Install pandas
```bash
pip install pandas
``` 


In [3]:
import pandas as pd
import re
from pathlib import Path

INPUT_FILE = Path("fined_list.jsonl")
OUTPUT_PATH = Path("news_in_txt")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

with INPUT_FILE.open('r', encoding='utf-8') as fin:
    df = pd.read_json(fin, lines=True)
for index, rows, in df.iterrows():
    OUTPUT_FILE = OUTPUT_PATH / f"{str(rows['date'])[:7]}_{rows['title']}.txt"
    with OUTPUT_FILE.open('w', encoding='utf-8') as fout:
        fout.write(rows['content'].strip()) 

## Tokenization and POS tagging



Install package spacy, type following scripts in terminal.
```bash
pip install -U spacy
python -m spacy download zh_core_web_trf
``` 

In [5]:
import pandas as pd
from pathlib import Path
import spacy
import re

INPUT_PATH = Path("news_in_txt")
OUTPUT_PATH = Path("newscontent_token_pos")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

nlp = spacy.load("zh_core_web_trf") 

rm_html = ['&', 'ens', 'p;'] #删除&ensp符号；

# for single_file in INPUT_PATH.glob("*.txt"):
for file in INPUT_PATH.glob("*.txt"):
    filename = file.stem 
    text = file.read_text()
    doc = nlp(text)
    tok_pos = [(f"{w.text}_{w.pos_}") for w in doc if w not in rm_html]  #排除html符号，遵格式要求打印

    OUTPUT_FILE = OUTPUT_PATH / f"{filename}_tokens_pos.txt"
    with OUTPUT_FILE.open('w', encoding='utf-8') as fout:
        fout.write(' '.join(tok_pos))

    print(f"The list has been written to {OUTPUT_FILE}")
print("Finish.")




The list has been written to newscontent_token_pos/2023-06_“小小议事员”沉浸式体验“全过程人民民主”_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2024-03_南城县人大：奏响“三部曲”深入践行全过程人民民主_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2021-11_充分发挥人大在全过程人民民主中的重要作用_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2022-05_“小快灵”背后的全过程人民民主_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2022-03_践行以人民为中心发展思想贯彻落实全过程人民民主_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2026-03_海门基层立法联系点举办两会精神宣讲会_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2023-04_重庆市沙坪坝区人大常委会基层立法联系点_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2024-03_以全过程人民民主汇聚奋进力量_tokens_pos.txt, 1 files in sum.
The list has been written to newscontent_token_pos/2023-03_湖南省长沙市人大常委会基层立法联系点_tokens_pos.txt, 1 files in sum.
